# Part B (PM) — Stretch: SVM for Text Classification (TF-IDF + SVM Pipeline)
**Day 33 | PM Session | Week 6**

We build a **TF-IDF + LinearSVC** pipeline on 4 newsgroup categories and compare it with Logistic Regression.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_20newsgroups
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.model_selection import cross_val_score

print('Libraries loaded.')

## 5 — Load 20 Newsgroups (4 Categories)

In [ ]:
CATEGORIES = [
    'sci.space',
    'rec.sport.hockey',
    'talk.politics.guns',
    'comp.graphics',
]

train_data = fetch_20newsgroups(subset='train', categories=CATEGORIES,
                                remove=('headers', 'footers', 'quotes'),
                                random_state=42)
test_data  = fetch_20newsgroups(subset='test',  categories=CATEGORIES,
                                remove=('headers', 'footers', 'quotes'),
                                random_state=42)

X_train_raw, y_train = train_data.data, train_data.target
X_test_raw,  y_test  = test_data.data,  test_data.target

print(f'Train docs : {len(X_train_raw)}')
print(f'Test  docs : {len(X_test_raw)}')
print(f'Categories : {CATEGORIES}')
print(f'\nSample text:\n{X_train_raw[0][:300]}...')

## 6 — Build TF-IDF + LinearSVC Pipeline

In [ ]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20_000,
        ngram_range=(1, 2),
        sublinear_tf=True,        # log-scale TF
        min_df=2
    )),
    ('clf', LinearSVC(C=1.0, max_iter=2000))
])

svm_pipeline.fit(X_train_raw, y_train)
y_pred_svm = svm_pipeline.predict(X_test_raw)

svm_acc = accuracy_score(y_test, y_pred_svm)
print(f'LinearSVC accuracy: {svm_acc:.4f}')

## 7 — Train & Evaluate

In [ ]:
print('=== LinearSVC Classification Report ===')
print(classification_report(y_test, y_pred_svm, target_names=CATEGORIES))

cm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.title('LinearSVC — Confusion Matrix', fontsize=13)
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 8 — Compare with Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2
    )),
    ('clf', LogisticRegression(C=5.0, max_iter=1000, solver='saga', n_jobs=-1))
])

lr_pipeline.fit(X_train_raw, y_train)
y_pred_lr = lr_pipeline.predict(X_test_raw)
lr_acc = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression accuracy: {lr_acc:.4f}')

print('\n=== Logistic Regression Classification Report ===')
print(classification_report(y_test, y_pred_lr, target_names=CATEGORIES))

In [ ]:
# Side-by-side accuracy bar chart
fig, ax = plt.subplots(figsize=(6, 4))
names  = ['LinearSVC', 'Logistic Regression']
accs   = [svm_acc, lr_acc]
colors = ['steelblue', 'darkorange']
bars   = ax.bar(names, accs, color=colors, edgecolor='k')
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylim(0.85, 1.0)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('TF-IDF: LinearSVC vs Logistic Regression\n(20Newsgroups — 4 categories)', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nConclusion:')
print('Both models perform very well on 4-class text classification.')
print('LinearSVC is typically faster (no probability calibration overhead).')
print('Logistic Regression gives calibrated probabilities — useful when confidence scores matter.')
print('SVM with linear kernel (LinearSVC) is the classic choice for text (sparse, high-dim features).')

## Why This Matters

SVM with a linear kernel (and its faster twin LinearSVC) is the **classic text classification algorithm** because:

1. **High-dimensional, sparse** — TF-IDF vectors have 10K-100K dimensions but mostly zeros. SVM's maximum-margin objective is extremely effective here.
2. **Only support vectors matter** — most documents aren't on the margin, so the model is compact.
3. **Industry-proven** — used at Google (spam), Yahoo News, Bloomberg (news categorisation).
4. **Fast at inference** — just a dot product between the query TF-IDF vector and a small set of weights.

> Today, fine-tuned Transformers (BERT, etc.) outperform TF-IDF + SVM, but for resource-constrained or latency-sensitive settings, TF-IDF + LinearSVC remains the gold standard.